# Notebook 02 - Yelp Data Preprocessing

This notebook specifically focuses on the data preprocessing code to help us filter our datasets before they are split for downstream use in training and evaluation. This part of the pipeline also includes our pick for k-core, filtering by open restaurants, features for our towers and feature-engineered choices, and splitting before save. 

### Filtering of Business Data

In [25]:
import pandas as pd
import json

yelp_business_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json"
with open(yelp_business_data_path, "r") as f:
    yelp_business_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to open restaurants/food businesses only
mask = (
    yelp_business_df["categories"].str.contains("Restaurants|Food", na=False) &
    (yelp_business_df["is_open"] == 1)
)
yelp_open_restaurants_df = yelp_business_df[mask].copy()

print(f"{len(yelp_open_restaurants_df):} open restaurant/food businesses")

44582 open restaurant/food businesses


### Filtration of Reviews by Business ID

In [26]:
yelp_review_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json"
with open(yelp_review_data_path, "r") as f:
    yelp_reviews_df = pd.DataFrame(json.loads(line) for line in f)

# Filter reviews to open restaurants only
open_restaurant_ids = set(yelp_open_restaurants_df["business_id"])
yelp_reviews_df = yelp_reviews_df[yelp_reviews_df["business_id"].isin(open_restaurant_ids)].copy()

print(f"{len(yelp_reviews_df):} reviews for open restaurants")

4104995 reviews for open restaurants


### K-Core Filtering

From the EDA notebook, we got the following table of values for the k-core statistics.  

| k-value | # of users | # of reviews | avg # of reviews/user|
| --- | --- | --- | --- |
| k = 5 | 210,133 users | 3,149,767 reviews | 15.0 avg reviews/user |
| k = 6 | 166,691 users | 2,932,557 reviews | 17.6 avg reviews/user |
| k = 7 | 136,534 users | 2,751,615 reviews | 20.2 avg reviews/user |

Based on the data, we decided to pick k = 5 as our k-core value. The reason behind picking k = 5  
was because we wanted to have a balanced tradeoff between the number of users, number of reviews,  
and the roughly average number of reviews/user.   

With a minimum baseline of 5 reviews for each user, this would also help combat reviews from low-activity users  
from affecting our recommendations significantly, ensuring that the quality of recommendations is up to a higher standard.

We are applying iterative k-core filtering to ensure every user has at least 5 reviews AND every business has at least 5 reviews simultaneously. Although the business JSON had a minimum of 5 reviews, after filtering to open restaurants only, some businesses lost reviews and dropped below k = 5. As previously mentioned in EDA, we expect to see slightly lower numbers than what the table states. 

In [27]:
k = 5
while True:
    user_counts = yelp_reviews_df["user_id"].value_counts()
    business_counts = yelp_reviews_df["business_id"].value_counts()

    val_users = set(user_counts[user_counts >= k].index)
    val_businesses = set(business_counts[business_counts >= k].index)

    k_filter_mask = yelp_reviews_df[
        yelp_reviews_df["user_id"].isin(val_users) &
        yelp_reviews_df["business_id"].isin(val_businesses)
    ]

    if len(k_filter_mask) == len(yelp_reviews_df):
        break

    yelp_reviews_df = k_filter_mask.copy()

print(f"{yelp_reviews_df['user_id'].nunique():} users, {yelp_reviews_df['business_id'].nunique():} businesses, {len(yelp_reviews_df):} reviews after k-core filtering")

166488 users, 39140 businesses, 2323580 reviews after k-core filtering


As evidenced in the results above, the number of users, businesses, and reviews in consideration reduced.  
However, roughly 166K users, 39K businesses, and 2.3M reviews is sufficient for creating meaningful training, validation, and test splits. 

### Filtering of Users (Matching with Reviews User IDs)

In [28]:
yelp_users_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_user.json"
with open(yelp_users_data_path, "r") as f:
    yelp_users_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to only users present in k-core filtered reviews
review_user_ids = set(yelp_reviews_df["user_id"])
yelp_users_df = yelp_users_df[yelp_users_df["user_id"].isin(review_user_ids)].copy()

# Checking if the user count matches
assert len(yelp_users_df) == yelp_reviews_df["user_id"].nunique(), "User count mismatch!"
print(f"User count matches: {len(yelp_users_df):} users")

User count matches: 166488 users
